# 第 2 章 — 遷移状態探索 (HCN ⇌ HNC, xTB)

**ゴール**
- xTB calculator を ASE 経由で使えるようになる
- 反応物・生成物を最小化して、TS 初期推定構造を組み立てる
- `Sella(order=1)` で 1 次鞍点を見つけ、**振動解析で虚振動 1 個** を確認する

**題材**: HCN ⇌ HNC のシス‐トランス的な異性化。教科書的な 1 次鞍点で、H が C–N 軸上をすべるように移動します。

## 事前準備

```bash
pip install tblite
```

`tblite` は GFN-xTB 系のメソッドを ASE calculator として提供します。CPU で十分高速です。

In [ ]:
from ase import Atoms
from tblite.ase import TBLite
from sella import Sella

def xtb():
    return TBLite(method='GFN2-xTB', verbosity=0)

# 反応物: HCN (直線)
hcn = Atoms('HCN', positions=[
    [-1.07, 0.0, 0.0],   # H
    [ 0.00, 0.0, 0.0],   # C
    [ 1.16, 0.0, 0.0],   # N
])
hcn.calc = xtb()
Sella(hcn, order=0, logfile=None).run(fmax=1e-3, steps=200)
E_hcn = hcn.get_potential_energy()
print(f'HCN  最適化後エネルギー: {E_hcn:.5f} eV')

In [ ]:
# 生成物: HNC (H が N 側についている)
hnc = Atoms('HCN', positions=[
    [ 2.16, 0.0, 0.0],   # H
    [ 0.00, 0.0, 0.0],   # C
    [ 1.16, 0.0, 0.0],   # N
])
hnc.calc = xtb()
Sella(hnc, order=0, logfile=None).run(fmax=1e-3, steps=200)
E_hnc = hnc.get_potential_energy()
print(f'HNC  最適化後エネルギー: {E_hnc:.5f} eV')
print(f'ΔE (HNC - HCN) = {(E_hnc - E_hcn)*1000:.1f} meV')

## TS 推定構造を組み立てる

HCN ⇌ HNC の TS は、**H が C と N の間を弧を描いて移る** 折れ曲がった構造です。  
C–N 結合長は 1.17 Å 程度のまま、H をだいたい等距離の弧上に置きます。

In [ ]:
import numpy as np

# C を原点、N を x 軸上、H を C-N の中点から y 方向に持ち上げた位置に置く
ts_guess = Atoms('HCN', positions=[
    [0.55, 1.10, 0.0],   # H (中点の上)
    [0.00, 0.00, 0.0],   # C
    [1.17, 0.00, 0.0],   # N
])
ts_guess.calc = xtb()
print(f'初期エネルギー: {ts_guess.get_potential_energy():.5f} eV')

## Sella(order=1) で鞍点を最適化

`order=1` を渡すと Sella は **1 個の負の固有値** を持つ点 (1 次鞍点) を探索します。

In [ ]:
opt = Sella(
    ts_guess,
    order=1,
    trajectory='ts.traj',
    logfile='ts.log',
)
opt.run(fmax=1e-3, steps=500)

E_ts = ts_guess.get_potential_energy()
print(f'TS エネルギー : {E_ts:.5f} eV')
print(f'順方向障壁 Ea(HCN→HNC) = {(E_ts - E_hcn):.3f} eV')
print(f'逆方向障壁 Ea(HNC→HCN) = {(E_ts - E_hnc):.3f} eV')

## 振動解析で TS を検証する

真の 1 次鞍点なら、**Hessian の固有値のうちちょうど 1 つが負** になります。  
ASE の `Vibrations` で確認します (虚振動は ASE では `0 - i * freq` のように複素表記)。

In [ ]:
import shutil, os
from ase.vibrations import Vibrations

# Vibrations は中間ファイルを 'vib/' に書き出す。再実行のため毎回消しておく
if os.path.isdir('vib'):
    shutil.rmtree('vib')

vib = Vibrations(ts_guess, name='vib/ts')
vib.run()
vib.summary()

# 周波数 (cm^-1) のうち虚数のものをカウント
freqs = vib.get_frequencies()  # 虚振動は複素数で返る
n_imag = int(sum(1 for f in freqs if np.iscomplex(f) and abs(f.imag) > 1e-3))
print(f'\n虚振動の数: {n_imag} (1 であれば 1 次鞍点)')

## 演習

1. TS 初期推定の `H` の位置 (`[0.55, 1.10, 0.0]`) を大きくずらしてみて、収束する／しないの境目を探してください。
2. `order=2` で実行したとき、どんな構造に落ちるか確認しましょう (2 次鞍点なので化学的には意味がないことが多いですが、Sella の挙動として勉強になります)。
3. `method='GFN1-xTB'` に変えるとエネルギーや障壁高さがどれくらい変わるか比較してください。

---
次章ではこの TS から IRC を流して、本当に HCN ⇄ HNC を繋いでいるかを確かめます。